# 02_Preprocessing.ipynb: Causal Time-Series Protocol & Feature Engineering
Tuân thủ nghiêm ngặt AGENTS.md và PROCESS.md: Không rò rỉ dữ liệu, bóc tách tính sẵn sàng (delay), walk-forward purging và đánh giá interval metrics trên khung cố định.

In [1]:
import pandas as pd
import numpy as np

def load_secom(data_path='data/secom/secom.data', labels_path='data/secom/secom_labels.data'):
    """Load raw SECOM dataset with strict validation on shape, labels, and stable alignment."""
    sensors = pd.read_csv(data_path, sep=r'\s+', header=None, dtype=np.float64)
    assert sensors.shape == (1567, 590), f'Unexpected sensor shape: {sensors.shape}'
    sensors.columns = [f'sensor_{i}' for i in range(sensors.shape[1])]
    
    labels = pd.read_csv(labels_path, sep=r'\s+', header=None, names=['raw_label', 'raw_time'])
    assert len(labels) == 1567, f'Unexpected labels count: {len(labels)}'
    assert set(labels['raw_label'].unique()).issubset({-1, 1}), 'Invalid labels outside {-1, 1}'
    
    ts = pd.to_datetime(labels['raw_time'].str.strip('"'), format='%d/%m/%Y %H:%M:%S')
    assert not ts.isna().any(), 'Found NaT in raw timestamps'
    
    df = pd.DataFrame({
        'timestamp': ts,
        'fail': labels['raw_label'].map({-1: 0, 1: 1}).astype(int),
        'raw_row_id': np.arange(len(labels))
    })
    assert df['fail'].sum() == 104, f'Expected 104 Fail records, got {df["fail"].sum()}'
    
    df = pd.concat([df, sensors], axis=1)
    # Stable sort preserving original row tie ordering
    df = df.sort_values(by=['timestamp', 'raw_row_id'], kind='mergesort').reset_index(drop=True)
    return df

In [2]:
def build_forecast_dataset(
    df, 
    lead=pd.Timedelta(hours=6),
    horizon=pd.Timedelta(hours=24),
    history=pd.Timedelta(hours=168),
    delay=pd.Timedelta(hours=6),
    cadence=pd.Timedelta(hours=6),
    q=0.10,
    min_event_samples=5,
    include_sensor_means=True
):
    """Build causal forecast origins after declared warmup up to dataset end minus lead minus horizon."""
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()
    
    min_ts = df['timestamp'].min()
    max_ts = df['timestamp'].max()
    
    # Warmup requirement: history must be available before first origin
    earliest_origin = (min_ts + history).ceil(cadence)
    latest_origin = (max_ts - lead - horizon).floor(cadence)
    
    if earliest_origin > latest_origin:
        return pd.DataFrame(), pd.DataFrame()
        
    origins = pd.date_range(start=earliest_origin, end=latest_origin, freq=cadence)
    feat_rows, meta_rows = [], []
    sensor_cols = [c for c in df.columns if c.startswith('sensor_')]
    
    for origin in origins:
        # Retrospective cut-off: records available at origin (timestamp + delay <= origin)
        avail_mask = (df['timestamp'] + delay <= origin) & (df['timestamp'] >= origin - history)
        avail_df = df.loc[avail_mask]
        
        f_dict = {}
        for w_h in [24, 72, 168]:
            sub = avail_df[avail_df['timestamp'] >= origin - pd.Timedelta(hours=w_h)]
            cnt = len(sub)
            f_dict[f'n_avail_{w_h}h'] = cnt
            f_dict[f'fail_rate_{w_h}h'] = sub['fail'].mean() if cnt > 0 else np.nan
            f_dict[f'missingness_{w_h}h'] = sub[sensor_cols].isna().mean().mean() if (cnt > 0 and len(sensor_cols) > 0) else np.nan
            
        f_dict['fail_rate_trend_24_vs_168'] = f_dict['fail_rate_24h'] - f_dict['fail_rate_168h']
        last_avail_ts = avail_df['timestamp'].max() if len(avail_df) > 0 else pd.NaT
        f_dict['hours_since_last_avail'] = (origin - last_avail_ts).total_seconds() / 3600.0 if pd.notna(last_avail_ts) else np.nan
        
        # Deterministic calendar features derived ONLY from known target_start = origin + lead
        # Known at origin time without peeking at actual observation times. No machine/shift ID assumed.
        t_start = origin + lead
        hod_frac = t_start.hour + t_start.minute / 60.0 + t_start.second / 3600.0
        dow_frac = t_start.dayofweek + hod_frac / 24.0
        f_dict['sin_hour_target_start'] = np.sin(2.0 * np.pi * hod_frac / 24.0)
        f_dict['cos_hour_target_start'] = np.cos(2.0 * np.pi * hod_frac / 24.0)
        f_dict['sin_dow_target_start'] = np.sin(2.0 * np.pi * dow_frac / 7.0)
        f_dict['cos_dow_target_start'] = np.cos(2.0 * np.pi * dow_frac / 7.0)
        
        if include_sensor_means and sensor_cols:
            sub_72 = avail_df[avail_df['timestamp'] >= origin - pd.Timedelta(hours=72)]
            means = sub_72[sensor_cols].mean() if len(sub_72) > 0 else pd.Series(np.nan, index=sensor_cols)
            for col, val in means.items():
                f_dict[f'{col}_mean72h'] = val
                
        feat_rows.append(f_dict)
        
        # Target window: [target_start, target_end) half-open
        target_start = origin + lead
        target_end = target_start + horizon
        target_avail = target_end + delay
        fut_obs = df[(df['timestamp'] >= target_start) & (df['timestamp'] < target_end)]
        n_fut = len(fut_obs)
        n_fail = int(fut_obs['fail'].sum())
        rate = (n_fail / n_fut) if n_fut > 0 else np.nan
        event_lbl = int(rate >= q) if (n_fut >= min_event_samples and pd.notna(rate)) else np.nan
        
        max_avail = (avail_df['timestamp'] + delay).max() if len(avail_df) > 0 else pd.NaT
        if pd.notna(max_avail):
            assert max_avail <= origin, 'Future sensor leakage: available timestamp exceeds origin!'
            
        meta_rows.append({
            'origin': origin,
            'target_start': target_start,
            'target_end': target_end,
            'target_available': target_avail,
            'n_future': n_fut,
            'n_fail': n_fail,
            'rate': rate,
            'event_label': event_lbl,
            'max_feature_available': max_avail
        })
        
    return pd.DataFrame(feat_rows, index=origins), pd.DataFrame(meta_rows, index=origins)

In [3]:
def training_mask(metadata, fit_time):
    """Purging & maturity rule: sample usable for training iff target_available <= fit_time."""
    mask = (metadata['target_available'] <= fit_time).to_numpy()
    return mask

def compute_incidence_matrix(raw_df, meta_subset, min_lead=pd.Timedelta(hours=6)):
    """Compute binary incidence matrix (n_contexts, n_records) matching causal lead and target window."""
    if len(meta_subset) == 0 or len(raw_df) == 0:
        return np.empty((len(meta_subset), len(raw_df)), dtype=bool)
    origins = meta_subset.index.values
    t_starts = meta_subset['target_start'].values
    t_ends = meta_subset['target_end'].values
    r_times = raw_df['timestamp'].values
    
    valid_lead = (r_times[None, :] >= (origins[:, None] + min_lead.to_timedelta64()))
    in_window = (r_times[None, :] >= t_starts[:, None]) & (r_times[None, :] < t_ends[:, None])
    return valid_lead & in_window

def capacity_threshold_at_origin(
    scores,
    metadata,
    raw_df,
    current_origin,
    model_ready_time,
    budget,
    lookback=pd.Timedelta(days=14),
    min_lead=pd.Timedelta(hours=6),
    n_quantiles=21,
    frozen_initial_threshold=None
):
    """
    Compute adaptive capacity threshold at current_origin using strictly completed historical contexts.
    Ignores non-finite scores (-inf, inf, nan). Never accesses future actual counts or labels.
    """
    current_origin = pd.Timestamp(current_origin)
    model_ready_time = pd.Timestamp(model_ready_time)
    
    assert len(scores) == len(metadata), 'scores and metadata must have same length'
    if not isinstance(scores, pd.Series):
        s_series = pd.Series(scores, index=metadata.index)
    else:
        s_series = scores

    hist_mask = (
        (metadata.index >= model_ready_time) &
        (metadata.index <= current_origin) &
        (metadata['target_available'] <= current_origin) &
        (metadata.index >= current_origin - lookback)
    )
    
    hist_meta = metadata.loc[hist_mask]
    hist_scores = s_series.loc[hist_mask].values
    
    if len(hist_meta) > 0:
        assert (hist_meta['target_available'] <= current_origin).all(), 'Past availability exceeds current origin!'
        last_label_available_time = hist_meta['target_available'].max()
    else:
        last_label_available_time = pd.NaT

    n_hist = len(hist_meta)
    if n_hist == 0:
        tau = float('inf') if frozen_initial_threshold is None else float(frozen_initial_threshold)
        return tau, {
            'last_label_available_time': last_label_available_time,
            'past_target_start_min': pd.NaT,
            'past_target_end_max': pd.NaT,
            'n_historical_contexts': 0,
            'n_past_raw_records': 0,
            'historical_workload': 0.0,
            'fallback_used': True,
            'reason': 'insufficient_history'
        }
        
    past_target_start = hist_meta['target_start'].min()
    past_target_end = hist_meta['target_end'].max()
    past_raw = raw_df[(raw_df['timestamp'] >= past_target_start) & (raw_df['timestamp'] < past_target_end)]
    n_past_raw = len(past_raw)
    
    if n_past_raw == 0:
        tau = float('inf') if frozen_initial_threshold is None else float(frozen_initial_threshold)
        return tau, {
            'last_label_available_time': last_label_available_time,
            'past_target_start_min': past_target_start,
            'past_target_end_max': past_target_end,
            'n_historical_contexts': n_hist,
            'n_past_raw_records': 0,
            'historical_workload': 0.0,
            'fallback_used': True,
            'reason': 'zero_past_raw_records'
        }
        
    finite_mask = np.isfinite(hist_scores)
    valid_scores = hist_scores[finite_mask]
    if len(valid_scores) == 0:
        # All scores non-finite (e.g. never_alert -inf) -> short-circuit to inf
        tau = float('inf') if frozen_initial_threshold is None else float(frozen_initial_threshold)
        return tau, {
            'last_label_available_time': last_label_available_time,
            'past_target_start_min': past_target_start,
            'past_target_end_max': past_target_end,
            'n_historical_contexts': n_hist,
            'n_past_raw_records': n_past_raw,
            'historical_workload': 0.0,
            'fallback_used': True,
            'reason': 'no_finite_scores'
        }

    q_grid = np.linspace(0.0, 1.0, n_quantiles)
    cand_thresholds = np.unique(np.quantile(valid_scores, q_grid))
    cand_thresholds = np.sort(np.append(cand_thresholds, float('inf')))
    
    inc = compute_incidence_matrix(past_raw, hist_meta, min_lead=min_lead)
    
    best_tau = float('inf')
    best_wl = -1.0
    
    for tau in cand_thresholds:
        warn_mask = (hist_scores >= tau)
        if not warn_mask.any():
            wl = 0.0
        else:
            rec_cov = inc[warn_mask].any(axis=0)
            wl = float(rec_cov.sum() / n_past_raw)
            
        if wl <= budget:
            if (wl > best_wl) or (np.isclose(wl, best_wl) and tau > best_tau):
                best_wl = wl
                best_tau = tau
                
    if best_wl < 0.0:
        best_tau = float('inf') if frozen_initial_threshold is None else float(frozen_initial_threshold)
        fallback = True
    else:
        fallback = False
        
    audit = {
        'last_label_available_time': last_label_available_time,
        'past_target_start_min': past_target_start,
        'past_target_end_max': past_target_end,
        'n_historical_contexts': n_hist,
        'n_past_raw_records': n_past_raw,
        'historical_workload': best_wl if not fallback else 0.0,
        'fallback_used': fallback,
        'reason': 'normal' if not fallback else 'no_cand_met_budget'
    }
    return float(best_tau), audit

def interval_metrics(df_raw, metadata, warnings_bool, eval_start, eval_end, min_lead=pd.Timedelta(hours=6)):
    """Evaluate alert coverage over fixed evaluation span [eval_start, eval_end)."""
    assert eval_start < eval_end, 'eval_start must be strictly earlier than eval_end'
    assert df_raw['raw_row_id'].is_unique, 'df_raw raw_row_id must be strictly unique'
    warn_arr = np.asarray(warnings_bool, dtype=bool)
    assert len(warn_arr) == len(metadata), f'warnings length {len(warn_arr)} != metadata {len(metadata)}'
    
    # Chronology and lead time assertions
    assert (metadata['origin'] < metadata['target_start']).all(), 'origin must precede target_start'
    assert (metadata['target_start'] < metadata['target_end']).all(), 'target_start must precede target_end'
    warn_idx = np.where(warn_arr)[0]
    if len(warn_idx) > 0:
        leads = metadata.iloc[warn_idx]['target_start'] - metadata.iloc[warn_idx]['origin']
        assert (leads >= min_lead).all(), 'Violated configured min_lead in warnings!'
        
    span_obs = df_raw[(df_raw['timestamp'] >= eval_start) & (df_raw['timestamp'] < eval_end)]
    total_fail = int(span_obs['fail'].sum())
    total_n = len(span_obs)
    
    covered_ids = set()
    clipped_intervals = []
    for i in warn_idx:
        s = max(metadata.iloc[i]['target_start'], eval_start)
        e = min(metadata.iloc[i]['target_end'], eval_end)
        if s < e:
            clipped_intervals.append((s, e))
            sub = span_obs[(span_obs['timestamp'] >= s) & (span_obs['timestamp'] < e)]
            covered_ids.update(sub['raw_row_id'].tolist())
            
    covered_obs = span_obs[span_obs['raw_row_id'].isin(covered_ids)]
    covered_fail = int(covered_obs['fail'].sum())
    covered_n = len(covered_obs)
    
    coverage = (covered_fail / total_fail) if total_fail > 0 else np.nan
    workload = (covered_n / total_n) if total_n > 0 else np.nan
    
    # Time under alert calculation
    clipped_intervals.sort()
    merged = []
    for iv in clipped_intervals:
        if not merged or merged[-1][1] < iv[0]:
            merged.append(list(iv))
        else:
            merged[-1][1] = max(merged[-1][1], iv[1])
    alert_seconds = sum((e - s).total_seconds() for s, e in merged)
    eval_span_seconds = (eval_end - eval_start).total_seconds()
    time_under_alert = (alert_seconds / eval_span_seconds) if eval_span_seconds > 0 else np.nan
    
    # Event metrics only on intervals fully contained within eval_start and eval_end
    contained = (metadata['target_start'] >= eval_start) & (metadata['target_end'] <= eval_end)
    valid_lbl = contained & (~metadata['event_label'].isna())
    tp = np.sum((warn_arr == 1) & (metadata['event_label'] == 1) & valid_lbl)
    fp = np.sum((warn_arr == 1) & (metadata['event_label'] == 0) & valid_lbl)
    fn = np.sum((warn_arr == 0) & (metadata['event_label'] == 1) & valid_lbl)
    prec = (tp / (tp + fp)) if (tp + fp) > 0 else np.nan
    rec = (tp / (tp + fn)) if (tp + fn) > 0 else np.nan
    
    return {
        'covered_fail': covered_fail,
        'total_fail': total_fail,
        'coverage': coverage,
        'covered_n': covered_n,
        'total_n': total_n,
        'workload': workload,
        'time_under_alert': time_under_alert,
        'event_precision': prec,
        'event_recall': rec
    }


In [4]:
# 1. Synthetic dataset setup
syn_dates = pd.date_range('2008-01-01 00:00:00', periods=12, freq='2h')
syn_df = pd.DataFrame({
    'timestamp': syn_dates,
    'fail': [0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0], # fails at 02:00, 06:00, 10:00
    'sensor_0': np.arange(12, dtype=float) * 1.5,
    'raw_row_id': np.arange(12)
})

# build dataset: lead=2h, horizon=4h, history=4h, delay=1h, cadence=2h
lead_s = pd.Timedelta(hours=2)
horizon_s = pd.Timedelta(hours=4)
hist_s = pd.Timedelta(hours=4)
delay_s = pd.Timedelta(hours=1)
cad_s = pd.Timedelta(hours=2)

X_syn, meta_syn = build_forecast_dataset(
    syn_df, lead=lead_s, horizon=horizon_s, history=hist_s,
    delay=delay_s, cadence=cad_s, q=0.1, min_event_samples=1, include_sensor_means=True
)

# Prechosen origin check for causal invariance (perturbing unavailable future sensors & labels)
target_origin = meta_syn.index[1]  # origin at 2008-01-01 06:00:00
syn_df_mod = syn_df.copy()
# Modify records strictly after origin - delay
syn_df_mod.loc[syn_df_mod['timestamp'] + delay_s > target_origin, 'fail'] = 1
syn_df_mod.loc[syn_df_mod['timestamp'] + delay_s > target_origin, 'sensor_0'] = 999.0
X_syn_mod, meta_syn_mod = build_forecast_dataset(
    syn_df_mod, lead=lead_s, horizon=horizon_s, history=hist_s,
    delay=delay_s, cadence=cad_s, q=0.1, min_event_samples=1, include_sensor_means=True
)
assert X_syn.loc[target_origin].equals(X_syn_mod.loc[target_origin]), 'Future invariance assertion failed!'
assert (meta_syn['max_feature_available'] <= meta_syn.index).all(), 'max_feature_available exceeds origin!'

# Exact synthetic metric tests with explicit min_lead
# Synthetic origin 0: origin=04:00, target=[06:00, 10:00)
# Synthetic origin 1: origin=06:00, target=[08:00, 12:00)
# Union of targets is [06:00, 12:00). Records at 06:00, 08:00, 10:00 (3 records). 12:00 is excluded (half-open).
# Fails in [06:00, 12:00) are at 06:00 and 10:00 (2 Fails).
warns = np.zeros(len(meta_syn), dtype=bool)
warns[0] = True; warns[1] = True
m_out = interval_metrics(syn_df, meta_syn, warns, eval_start=pd.Timestamp('2008-01-01 00:00:00'), eval_end=pd.Timestamp('2008-01-01 22:00:00'), min_lead=lead_s)
assert m_out['covered_n'] == 3, f'Expected 3 covered records, got {m_out["covered_n"]}'
assert m_out['covered_fail'] == 2, f'Expected 2 covered fails, got {m_out["covered_fail"]}'
assert m_out['total_fail'] == 3, f'Expected 3 total fails, got {m_out["total_fail"]}'
assert np.isclose(m_out['coverage'], 2/3), 'Coverage mismatch on exact overlap'

# Single warned origin test: [06:00, 10:00) contains 06:00, 08:00 (2 records, 1 Fail at 06:00)
warns_single = np.zeros(len(meta_syn), dtype=bool); warns_single[0] = True
m_single = interval_metrics(syn_df, meta_syn, warns_single, eval_start=pd.Timestamp('2008-01-01 00:00:00'), eval_end=pd.Timestamp('2008-01-01 22:00:00'), min_lead=lead_s)
assert m_single['covered_n'] == 2 and m_single['covered_fail'] == 1

# Zero Fail metric test: coverage must be NaN
syn_no_fail = syn_df.copy(); syn_no_fail['fail'] = 0
m_zero = interval_metrics(syn_no_fail, meta_syn, warns, eval_start=pd.Timestamp('2008-01-01 00:00:00'), eval_end=pd.Timestamp('2008-01-01 22:00:00'), min_lead=lead_s)
assert np.isnan(m_zero['coverage']), 'Zero-fail coverage must be NaN'

# 2. Dev smoke run
df_full = load_secom()
dev_cutoff = pd.Timestamp('2008-10-02 20:54:00')
df_dev = df_full[df_full['timestamp'] < dev_cutoff].copy()

X_dev, meta_dev = build_forecast_dataset(df_dev, include_sensor_means=False)
mask_train = training_mask(meta_dev, fit_time=pd.Timestamp('2008-08-01 00:00:00'))

assert 0 < mask_train.sum() < len(mask_train), 'Maturity mask truth is trivial!'
print('All synthetic tests and causal checks passed.')
print(f'df_dev shape: {df_dev.shape}')
print(f'X_dev shape: {X_dev.shape}, meta_dev shape: {meta_dev.shape}')
print(f'Event label counts:\n{meta_dev["event_label"].value_counts(dropna=False)}')
print(f'Maturity mask sum at fit_time: {mask_train.sum()} / {len(mask_train)}')


# Future-invariance test for capacity_threshold_at_origin
syn_scores_policy = pd.Series(np.linspace(0.1, 0.9, len(meta_syn)), index=meta_syn.index)
current_test_origin = meta_syn.index[1]
ready_test_time = meta_syn.index[0]

tau_orig, audit_orig = capacity_threshold_at_origin(
    syn_scores_policy, meta_syn, syn_df,
    current_origin=current_test_origin,
    model_ready_time=ready_test_time,
    budget=0.30,
    lookback=pd.Timedelta(hours=12),
    min_lead=lead_s
)

# Perturb future raw records (strictly >= current_test_origin)
syn_df_perturbed = syn_df.copy()
future_mask = syn_df_perturbed['timestamp'] >= current_test_origin
syn_df_perturbed.loc[future_mask, 'fail'] = 1 - syn_df_perturbed.loc[future_mask, 'fail']
# Append extra future rows
extra_future = pd.DataFrame({
    'timestamp': pd.date_range(syn_dates[-1] + pd.Timedelta(hours=2), periods=5, freq='2h'),
    'fail': [1, 1, 1, 1, 1],
    'sensor_0': np.zeros(5),
    'raw_row_id': np.arange(12, 17)
})
syn_df_perturbed = pd.concat([syn_df_perturbed, extra_future], ignore_index=True)

# Perturb future scores strictly > current_test_origin
syn_scores_perturbed = syn_scores_policy.copy()
future_origins = meta_syn.index[meta_syn.index > current_test_origin]
syn_scores_perturbed.loc[future_origins] = 999.0

tau_pert, audit_pert = capacity_threshold_at_origin(
    syn_scores_perturbed, meta_syn, syn_df_perturbed,
    current_origin=current_test_origin,
    model_ready_time=ready_test_time,
    budget=0.30,
    lookback=pd.Timedelta(hours=12),
    min_lead=lead_s
)

assert tau_orig == tau_pert, f'Future invariance failed: {tau_orig} != {tau_pert}'
assert audit_orig == audit_pert, 'Audit mismatch under future perturbation'
print('capacity_threshold_at_origin future-invariance test passed.')


All synthetic tests and causal checks passed.
df_dev shape: (1253, 593)
X_dev shape: (269, 15), meta_dev shape: (269, 9)
Event label counts:
event_label
0.0    157
1.0     72
NaN     40
Name: count, dtype: int64
Maturity mask sum at fit_time: 17 / 269
capacity_threshold_at_origin future-invariance test passed.
